In [ ]:
# =============================================
# 1. IMPORTE
# =============================================

import pandas as pd
import numpy as np
import random
import torch
from sklearn.model_selection import train_test_split
from transformers import pipeline
import spacy
from datasets import Dataset, DatasetDict
import re
from tqdm import tqdm

# Setze alle Random-Seeds für Reproduzierbarkeit
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Lade spaCy-Modell für NER, POS, Stopwörter
try:
    nlp = spacy.load("de_core_news_md")
except OSError:
    print("Downloading spaCy model 'de_core_news_md'...")
    spacy.cli.download("de_core_news_md")
    nlp = spacy.load("de_core_news_md")

# Lade BERT-Pipeline für Masked Language Modeling
fill_mask_pipeline = pipeline(
    "fill-mask",
    model="dbmdz/bert-base-german-cased",
    top_k=3,
    device=0 if torch.cuda.is_available() else -1  # Nutze GPU, falls verfügbar
)

# =============================================
# 2. KONFIGURATION
# =============================================

# Augmentierungsparameter
#AUGMENT_RATIO = 0.20  # 20% der Trainingsdaten werden augmentiert
AUGMENT_RATIO = 1  # 100% der Trainingsdaten werden augmentiert
TOP_K = 3             # Anzahl BERT-Vorschläge pro Maskierung
MIN_TOKEN_LENGTH = 5  # Mindestlänge für Kandidatenwörter
MIN_SCORE = 0.20     # Mindestwahrscheinlichkeit für BERT-Vorschläge

# Geschützte Entity-Typen (spaCy NER)
PROTECTED_ENTITIES = {"PER", "ORG", "LOC", "GPE", "MISC"}

# Erlaubte POS-Tags für Ersatz (spaCy)
ALLOWED_POS = {"NOUN", "ADJ"}

# =============================================
# 3. DATEN LADEN UND VORBEREITEN
# =============================================

# Lade den Original-Datensatz
file_path = "ground_truth_LLM804.xlsx"
df = pd.read_excel(file_path)

# Überprüfe, ob die Spalte 'Headline_GER' existiert
if "Headline_GER" not in df.columns:
    raise ValueError("Die Spalte 'Headline_GER' wurde nicht in der Excel-Datei gefunden.")

# Identifiziere alle Label-Spalten (Annahme: Alle Spalten außer 'Headline_GER' sind Labels)
label_columns = [col for col in df.columns if col != "Headline_GER"]

# Convert the label columns into a single list of floats for each row
df['labels'] = df[label_columns].values.astype(np.float32).tolist()

# Keep only the necessary columns for training
df_clean = df[["Headline_GER", 'labels']].copy()

# =============================================
# 4. DATENSPLITTING (Train/Dev/Test)
# =============================================

# Initial 80/20 Train-Rest split
train_df, rest_df = train_test_split(df_clean, test_size=0.20, random_state=SEED)

# Splitting the Rest (20%) into 60% Dev and 40% Test
dev_df, test_df = train_test_split(rest_df, test_size=0.40, random_state=SEED)

# Füge 'split'-Spalte hinzu (for consistency, though not strictly needed for the split itself)
train_df["split"] = "train"
dev_df["split"] = "dev"
test_df["split"] = "test"

# Convert Pandas DataFrames into Hugging Face Datasets
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev': Dataset.from_pandas(dev_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

# =============================================
# 5. HILFSFUNKTIONEN FÜR AUGMENTIERUNG
# =============================================

def is_protected_entity(token):
    """Prüfe, ob ein Token eine geschützte Named Entity ist."""
    return token.ent_type_ in PROTECTED_ENTITIES

def is_abbreviation(token):
    """Prüfe, ob ein Token eine Abkürzung (alle Großbuchstaben) ist."""
    return token.text.isupper() and len(token.text) > 1

def is_number(token):
    """Prüfe, ob ein Token eine Zahl enthält."""
    return any(char.isdigit() for char in token.text)

def is_special_char(token):
    """Prüfe, ob ein Token ein Sonderzeichen ist."""
    return any(char in token.text for char in {"%", "&", "€", "@", "#"})

def has_min_length(token):
    """Prüfe, ob ein Token die Mindestlänge hat."""
    return len(token.text) >= MIN_TOKEN_LENGTH

def is_allowed_pos(token):
    """Prüfe, ob ein Token eine erlaubte Wortart hat."""
    return token.pos_ in ALLOWED_POS

def get_candidate_tokens(doc):
    """Extrahiere Kandidaten-Tokens für die Maskierung."""
    candidates = []
    for token in doc:
        if (not is_protected_entity(token) and
            not is_abbreviation(token) and
            not is_number(token) and
            not is_special_char(token) and
            not token.is_stop and
            has_min_length(token) and
            is_allowed_pos(token)):
            candidates.append(token)
    return candidates

def matches_case_pattern(original, replacement):
    """Prüfe, ob das Ersatzwort das gleiche Groß-/Kleinschreibungsmuster hat."""
    if original.istitle():
        return replacement.istitle()
    elif original.isupper():
        return replacement.isupper()
    elif original.islower():
        return replacement.islower()
    else:
        return True  # Gemischte Schreibweise (z. B. "iPhone") wird nicht geprüft

def is_wordpiece(token):
    """Prüfe, ob ein Token ein WordPiece-Fragment (z. B. '##ung') ist."""
    return token.startswith("##")

def is_valid_replacement(original_token, replacement, stop_words):
    """Prüfe, ob ein Ersatzwort alle Qualitätsfilter besteht."""
    if (replacement.lower() == original_token.text.lower() or
        is_wordpiece(replacement) or
        replacement in stop_words or
        len(replacement) < MIN_TOKEN_LENGTH or
        any(char.isdigit() for char in replacement) or
        not matches_case_pattern(original_token.text, replacement)):
        return False
    return True

# =============================================
# 6. AUGMENTIERUNG DES TRAININGSSATZES (mit Batch-Verarbeitung)
# =============================================

# Lade spaCy-Stopwörter
stop_words = nlp.Defaults.stop_words

# Wähle zufällig einen Teil der Trainingsdaten für die Augmentierung aus
num_to_augment = int(len(train_df) * AUGMENT_RATIO)
# Ensure we don't try to sample more rows than available in train_df
if num_to_augment > len(train_df):
    num_to_augment = len(train_df)

train_df_to_augment = train_df.sample(n=num_to_augment, random_state=SEED)

masked_headlines_info = [] # Speichert {original_row, original_headline, selected_token, masked_headline_str}

# Schritt 1: Sammle alle maskierten Headlines und die notwendigen Informationen
# für die spätere Rekonstruktion
print("Schritt 1/3: Maskierte Headlines vorbereiten...")
for idx, row in tqdm(train_df_to_augment.iterrows(), total=len(train_df_to_augment), desc="Vorbereitung Maskierung"):
    headline = row["Headline_GER"]
    doc = nlp(headline)
    candidates = get_candidate_tokens(doc)

    if candidates:
        random.shuffle(candidates)
        selected_token = candidates[0]
        # Ersetzen des ausgewählten Tokens durch das Masken-Token
        masked_headline_str = headline.replace(selected_token.text, "[MASK]", 1)

        masked_headlines_info.append({
            "original_row": row,
            "original_headline": headline,
            "selected_token": selected_token,
            "masked_headline_str": masked_headline_str
        })

if not masked_headlines_info:
    print("Warnung: Keine gültigen Kandidaten für die Maskierung in den ausgewählten Headlines gefunden.")
    augmented_data = [] # Initialisiere leere Liste, wenn keine Augmentierung möglich ist
else:
    # Schritt 2: Führe Batch-Inferenz für alle maskierten Headlines durch
    masked_headlines_list = [item["masked_headline_str"] for item in masked_headlines_info]
    print(f"Schritt 2/3: Führe Batch-Maskierung für {len(masked_headlines_list)} Headlines durch...")
    # Die Pipeline unterstützt einen batch_size-Parameter für effiziente GPU-Nutzung
    batch_predictions = fill_mask_pipeline(masked_headlines_list, batch_size=16) # Batch-Größe anpassen bei Bedarf

    # Schritt 3: Verarbeite die Batch-Vorschläge und rekonstruiere augmentierte Headlines
    augmented_data = []
    print("Schritt 3/3: Augmentierte Headlines rekonstruieren...")
    for i, preds_for_one_headline in tqdm(enumerate(batch_predictions), total=len(batch_predictions), desc="Verarbeitung Batch-Vorschläge"):
        item = masked_headlines_info[i]
        original_row = item["original_row"]
        original_headline = item["original_headline"]
        selected_token = item["selected_token"] # spaCy token-Objekt

        for pred in preds_for_one_headline:
            replacement_str = pred["token_str"]
            score = pred["score"]

            if score < MIN_SCORE:
                continue

            # is_valid_replacement erwartet ein spaCy-Token für 'original_token'
            # und einen String für 'replacement'
            if is_valid_replacement(selected_token, replacement_str, stop_words):
                augmented_headline = original_headline.replace("[MASK]", replacement_str, 1)
                new_row = original_row.copy()
                new_row["Headline_GER"] = augmented_headline
                new_row["is_augmented"] = 1
                augmented_data.append(new_row)
                break # Gültiger Ersatz für diese Headline gefunden, weiter zur nächsten

    if not augmented_data:
        print("Warnung: Keine gültigen Augmentierungen nach Filterung der BERT-Vorschläge gefunden!")

# Füge augmentierte Daten zum Trainingsset hinzu
if augmented_data:
    augmented_df = pd.DataFrame(augmented_data)
    train_df = pd.concat([train_df, augmented_df], ignore_index=True)
else:
    print("Warnung: Keine augmentierten Daten zum Trainingsset hinzugefügt.")

# Füge 'is_augmented'-Spalte für nicht-augmentierte Daten hinzu
train_df["is_augmented"] = train_df["is_augmented"].fillna(0).astype(int)
dev_df["is_augmented"] = 0
test_df["is_augmented"] = 0

# =============================================
# 7. ZUSAMMENFÜHREN UND SPEICHERN
# =============================================

# Kombiniere alle Daten
final_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)

# Speichere als Excel-Datei
output_file = "ground_truth_LLM804_context-augmented_1.xlsx"
final_df.to_excel(output_file, index=False)
print(f"Augmentierter Datensatz gespeichert unter: {output_file}")

# =============================================
# 8. ERSTELLE HUGGINGFACE DATASETDICT
# =============================================

# Konvertiere DataFrames zu HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)

# Erstelle DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "dev": dev_dataset,
    "test": test_dataset
})

print("DatasetDict erstellt.")



from google.colab import files

# Datei herunterladen
files.download(output_file)

In [ ]:
import ast
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ---- 1. Konfiguration & Modell-ID Setup

model_id = "dbmdz/bert-base-german-cased"  # Passe die Modell-ID bei Bedarf an
num_labels = 14

tokenizer = AutoTokenizer.from_pretrained(model_id)

config = AutoConfig.from_pretrained(
    model_id,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

# ---- 2. Daten laden und aufbereiten

df = pd.read_excel('ground_truth_LLM804_context-augmented_1.xlsx')
text_col = 'Headline_GER'

# Konvertiere den String der Liste in echte Python-Listen aus Floats
df['labels'] = df['labels'].apply(lambda x: [float(i) for i in ast.literal_eval(x)] if isinstance(x, str) else x)

# Nutzung der vordefinierten Splits
train_df = df[df['split'] == 'train'][[text_col, 'labels']].copy()
dev_df   = df[df['split'] == 'dev'][[text_col, 'labels']].copy()
test_df  = df[df['split'] == 'test'][[text_col, 'labels']].copy()

print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

# Konvertierung in Hugging Face Datasets
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'dev':   Dataset.from_pandas(dev_df, preserve_index=False),
    'test':  Dataset.from_pandas(test_df, preserve_index=False)
})

# ---- 3. Tokenisierung & Metriken

def tokenize_function(examples):
    return tokenizer(
        examples[text_col],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Sigmoid für Multi-Label-Klassifikation
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    f1_micro = f1_score(labels, preds, average='micro', zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    
    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
    }

# ---- 4. TrainingArguments & Modell-Initialisierung

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=10,                 # Auf 10 Epochen erhöht
    weight_decay=0.01,
    load_best_model_at_end=True,         # Lädt automatisch den besten Checkpoint nach dem Training
    metric_for_best_model='f1_macro',    # Bewertungsmetrik für das beste Modell
    greater_is_better=True,
    save_total_limit=1,                  # Speichert Platz, indem nur der beste Checkpoint auf der Festplatte bleibt
    logging_steps=10
)

model_LLM = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    config=config
)

# ---- 5. Trainer Setup, Training und Evaluation

trainer = Trainer(
    model=model_LLM,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    compute_metrics=compute_metrics
)

# Trainingsloop ausführen (über 10 Epochen)
trainer.train()

# Speichert den am besten evaluierten Modellzustand in einem separaten Ordner
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

# Finale Evaluation gegen das ungesehene Test-Set mit dem besten Modell
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)